# R3-I: Expected calibration error (ECE) only
Attach the dataset **snapshot-19-11-2026**. CPU, no Internet needed. Run all.
Output: /kaggle/working/R3_results.zip with R3_I_ece.json.

In [ ]:
# ============================== SETUP ==============================
SMOKE = False            # True = small quick test (minutes)
RUN = dict(A=False, B=False, C=False, D=False, E=False, F=False, G=False, H=False, I=True)  # A-G already done
H_STRATEGIES = ("limit", "kfactor", "kfactor_neg", "mc")  # part H inverter fault strategies
H_PEN = 0.6              # part H: inverter output, share of feeder load
H_IMAX = 1.2             # part H: inverter current limit (pu of rated current)
N_SIM = 12000            # samples per new simulation (parts F, G); same as the original data sets
RF_MAX_NEW = 1000.0      # part F: upper fault resistance (ohm)
DER_LEVELS = (0.3, 0.6)  # part G: inverter DER penetration (share of feeder load)

import os, sys, glob, json, time, shutil, warnings
warnings.filterwarnings("ignore")
t0 = time.time()
_c = sorted(glob.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
assert _c, "Attach the dataset 'snapshot-fd-code-data'"
DATA = os.path.dirname(_c[0])
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results"); os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS; os.environ["K4_DATA"] = DATA
sys.path.insert(0, DATA)

if RUN["F"] or RUN["G"] or RUN.get("H"):
    try:
        import opendssdirect  # noqa
    except ImportError:
        whl = sorted(glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True))
        if whl:   # offline: dataset 'opendss-wheels'
            os.system("pip install -q --no-index --no-deps " + " ".join(f'"{w}"' for w in whl))
        else:     # online: needs Internet ON
            os.system("pip install -q OpenDSSDirect.py")
try:
    import opendssdirect  # noqa
    HAVE_DSS = True
except ImportError:
    import types; sys.modules["opendssdirect"] = types.ModuleType("opendssdirect"); HAVE_DSS = False

import numpy as np, pandas as pd
import k4lib as K, features as F, models as M, placement2 as Q
from sklearn.base import clone
print("DATA", DATA, "| OpenDSS", HAVE_DSS, "| 1% TVE noise", K.NOISE["1% TVE"])

SEEDS = (0,) if SMOKE else (0, 1)
FEEDERS = [13] if SMOKE else [13, 34, 123]
NZ = "1% TVE"

def save(name, obj):
    with open(os.path.join(RESULTS, name + ".json"), "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    print("saved", name)

def small(ds, n=1500):
    # SMOKE: keep a stratified subset
    if not SMOKE: return ds
    idx = np.random.RandomState(0).permutation(ds.n)[:n]
    for a in ("Vpre", "Vf", "Ipre", "If", "ftype", "fbus", "rf", "y"):
        setattr(ds, a, getattr(ds, a)[idx])
    ds.n = len(idx); return ds

def dmp_sets(nb, k=3):
    # the sensor sets used in the manuscript (Table 7), recorded by e14
    b = K.baselines(nb, k)
    if "DMP (proposed)" in b:
        return {"DMP": b["DMP (proposed)"]["buses"], "DMP N-1": b["DMP N-1 (proposed)"]["buses"]}
    raise RuntimeError("e14_placement_baselines.json not found in the dataset")

PAPER_3BUS = ["650", "675", "684"]

## I. Expected calibration error (comment C6: ECE is listed in Fig. 2 but not reported)
ECE = sum over 15 confidence bins of (share of snapshots in the bin) x |accuracy - mean confidence|,
with confidence = largest predicted class probability (same definition as e15).
DMP three-sensor sets, LightGBM, 2 seeds x 5 folds, AGG / SEQ / FPS, 1% TVE and micro-PMU.
Also reported: accuracy of the 10% least confident snapshots, which links ECE to the reject option.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
def ece(proba, ok, bins=15):
    conf = proba.max(1); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            e += s.mean() * abs(ok[s].mean() - conf[s].mean())
    return float(e)

if RUN.get("I"):
    rows = []
    for nb in FEEDERS:
        ds = small(K.dataset(nb)); obs = K.rows_for_buses(ds, dmp_sets(nb)["DMP"])
        for noise in ("1% TVE", "uPMU"):
            for rep in ("AGG", "SEQ", "FPS"):
                X = K.features_of(ds, obs, 0, noise, rep)
                eces, accs, low10 = [], [], []
                for s_ in SEEDS:
                    for tr, te in StratifiedKFold(5, shuffle=True, random_state=s_).split(X, ds.y):
                        m = LGBMClassifier(**K.FAST).fit(X[tr], ds.y[tr])
                        pr = m.predict_proba(X[te]); ok = pr.argmax(1) == ds.y[te]
                        eces.append(ece(pr, ok)); accs.append(ok.mean())
                        cut = np.quantile(pr.max(1), 0.10); low10.append(ok[pr.max(1) <= cut].mean())
                rows.append(dict(feeder=nb, noise=noise, rep=rep, ece=float(np.mean(eces)), ece_sd=float(np.std(eces)),
                                 acc=float(np.mean(accs)), acc_low10=float(np.mean(low10))))
                print(nb, noise, rep, f"ECE {100*np.mean(eces):.2f} +- {100*np.std(eces):.2f} %  acc {100*np.mean(accs):.1f}"
                      f"  acc of 10% least confident {100*np.mean(low10):.1f}", flush=True)
    save("R3_I_ece", rows)

In [ ]:
shutil.make_archive(os.path.join(WORK, "R3_results"), "zip", RESULTS)
print("wrote /kaggle/working/R3_results.zip   total", round((time.time() - t0) / 3600, 2), "h")